<a href="https://colab.research.google.com/github/leejunho12316/HonGongMachine/blob/main/Function_Calling_LangChain_%EC%97%86%EC%9D%B4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import json
import pandas as pd
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Union
from openai import OpenAI
import gradio as gr

In [2]:
# 샘플 데이터 (실제 구현에서는 DB나 API에서 가져올 것입니다)
destinations_data = [
    {"목적지ID": "D001", "도시": "파리", "국가": "프랑스", "대륙": "유럽", "언어": "프랑스어", "시차": 7, "통화": "유로", "기후": "온대", "대표음식": "크루아상, 에스카르고"},
    {"목적지ID": "D002", "도시": "도쿄", "국가": "일본", "대륙": "아시아", "언어": "일본어", "시차": -1, "통화": "엔", "기후": "온대", "대표음식": "스시, 라멘"},
    {"목적지ID": "D003", "도시": "뉴욕", "국가": "미국", "대륙": "북미", "언어": "영어", "시차": 13, "통화": "달러", "기후": "대륙성", "대표음식": "핫도그, 베이글"},
    {"목적지ID": "D004", "도시": "방콕", "국가": "태국", "대륙": "아시아", "언어": "태국어", "시차": -2, "통화": "바트", "기후": "열대", "대표음식": "팟타이, 똠양꿍"},
    {"목적지ID": "D005", "도시": "로마", "국가": "이탈리아", "대륙": "유럽", "언어": "이탈리아어", "시차": 7, "통화": "유로", "기후": "지중해성", "대표음식": "파스타, 피자"}
]

attractions_data = [
    {"관광지ID": "A001", "목적지ID": "D001", "명소명": "에펠탑", "종류": "랜드마크", "입장료": 25, "추천기간": "봄,가을", "방문시간": 3, "인기도": 5},
    {"관광지ID": "A002", "목적지ID": "D001", "명소명": "루브르 박물관", "종류": "박물관", "입장료": 17, "추천기간": "전체", "방문시간": 4, "인기도": 5},
    {"관광지ID": "A003", "목적지ID": "D001", "명소명": "몽마르뜨", "종류": "지역", "입장료": 0, "추천기간": "봄,여름", "방문시간": 3, "인기도": 4},
    {"관광지ID": "A004", "목적지ID": "D002", "명소명": "도쿄 스카이트리", "종류": "랜드마크", "입장료": 23, "추천기간": "전체", "방문시간": 2, "인기도": 4},
    {"관광지ID": "A005", "목적지ID": "D002", "명소명": "센소지 사원", "종류": "종교시설", "입장료": 0, "추천기간": "봄", "방문시간": 1, "인기도": 4},
    {"관광지ID": "A006", "목적지ID": "D002", "명소명": "시부야 교차로", "종류": "랜드마크", "입장료": 0, "추천기간": "전체", "방문시간": 1, "인기도": 5},
    {"관광지ID": "A007", "목적지ID": "D003", "명소명": "자유의 여신상", "종류": "랜드마크", "입장료": 24, "추천기간": "봄,가을", "방문시간": 3, "인기도": 5},
    {"관광지ID": "A008", "목적지ID": "D003", "명소명": "타임스퀘어", "종류": "지역", "입장료": 0, "추천기간": "전체", "방문시간": 2, "인기도": 5},
    {"관광지ID": "A009", "목적지ID": "D004", "명소명": "왕궁", "종류": "역사유적", "입장료": 15, "추천기간": "겨울", "방문시간": 3, "인기도": 4},
    {"관광지ID": "A010", "목적지ID": "D005", "명소명": "콜로세움", "종류": "역사유적", "입장료": 16, "추천기간": "봄,가을", "방문시간": 2, "인기도": 5}
]

hotels_data = [
    {"호텔ID": "H001", "목적지ID": "D001", "호텔명": "호텔 드 루브르", "등급": 4, "가격대": "고가", "위치": "시내중심", "평점": 4.5, "편의시설": "수영장,레스토랑,와이파이"},
    {"호텔ID": "H002", "목적지ID": "D001", "호텔명": "파리 백팩커스", "등급": 2, "가격대": "저가", "위치": "교외", "평점": 3.8, "편의시설": "와이파이,조식"},
    {"호텔ID": "H003", "목적지ID": "D002", "호텔명": "도쿄 프라자", "등급": 5, "가격대": "고가", "위치": "시내중심", "평점": 4.7, "편의시설": "수영장,스파,레스토랑,와이파이,헬스장"},
    {"호텔ID": "H004", "목적지ID": "D002", "호텔명": "아키하바라 인", "등급": 3, "가격대": "중가", "위치": "관광지인접", "평점": 4.0, "편의시설": "와이파이,조식,레스토랑"},
    {"호텔ID": "H005", "목적지ID": "D003", "호텔명": "맨해튼 그랜드", "등급": 5, "가격대": "고가", "위치": "시내중심", "평점": 4.6, "편의시설": "수영장,스파,레스토랑,와이파이,헬스장,바"}
]

flights_data = [
    {"항공편ID": "F001", "출발지": "서울", "도착지": "파리", "항공사": "대한항공", "출발시간": "08:30", "도착시간": "14:25", "가격": 1200000, "비행시간": 13, "경유": 0},
    {"항공편ID": "F002", "출발지": "서울", "도착지": "파리", "항공사": "에어프랑스", "출발시간": "13:45", "도착시간": "18:35", "가격": 1150000, "비행시간": 12, "경유": 1},
    {"항공편ID": "F003", "출발지": "서울", "도착지": "도쿄", "항공사": "대한항공", "출발시간": "09:20", "도착시간": "11:40", "가격": 350000, "비행시간": 2, "경유": 0},
    {"항공편ID": "F004", "출발지": "서울", "도착지": "도쿄", "항공사": "일본항공", "출발시간": "15:30", "도착시간": "17:45", "가격": 320000, "비행시간": 2, "경유": 0},
    {"항공편ID": "F005", "출발지": "서울", "도착지": "뉴욕", "항공사": "대한항공", "출발시간": "10:00", "도착시간": "10:30", "가격": 1500000, "비행시간": 14, "경유": 0}
]

packages_data = [
    {"패키지ID": "P001", "목적지ID": "D001", "패키지명": "파리 3박4일 에펠탑 투어", "가격": 1500000, "기간": 4, "포함사항": "호텔,항공,관광,식사 2회", "제외사항": "개인경비,팁", "인기도": 4},
    {"패키지ID": "P002", "목적지ID": "D001", "패키지명": "파리 5박6일 루브르 특별 투어", "가격": 2100000, "기간": 6, "포함사항": "호텔,항공,관광,식사 5회", "제외사항": "개인경비,팁", "인기도": 5},
    {"패키지ID": "P003", "목적지ID": "D002", "패키지명": "도쿄 2박3일 빠른여행", "가격": 800000, "기간": 3, "포함사항": "호텔,항공,관광", "제외사항": "식사,개인경비,팁", "인기도": 3},
    {"패키지ID": "P004", "목적지ID": "D002", "패키지명": "일본 도쿄 4박5일 맛집 투어", "가격": 1200000, "기간": 5, "포함사항": "호텔,항공,관광,식사 8회", "제외사항": "개인경비,팁", "인기도": 5},
    {"패키지ID": "P005", "목적지ID": "D003", "패키지명": "뉴욕 핵심 관광 6박7일", "가격": 2500000, "기간": 7, "포함사항": "호텔,항공,관광,식사 3회", "제외사항": "개인경비,팁", "인기도": 4}
]

user_trips_data = [
    {"여행ID": "T001", "사용자ID": "U001", "목적지ID": "D001", "출발일": "2025-05-15", "귀국일": "2025-05-19", "예산": 2000000, "상태": "계획중"},
    {"여행ID": "T002", "사용자ID": "U001", "목적지ID": "D002", "출발일": "2024-07-20", "귀국일": "2024-07-25", "예산": 1500000, "상태": "완료"},
    {"여행ID": "T003", "사용자ID": "U002", "목적지ID": "D003", "출발일": "2025-06-10", "귀국일": "2025-06-16", "예산": 3000000, "상태": "계획중"}
]

trip_items_data = [
    {"항목ID": "I001", "여행ID": "T001", "종류": "숙소", "세부내용": "호텔 드 루브르", "예약번호": "H123456", "비용": 650000},
    {"항목ID": "I002", "여행ID": "T001", "종류": "항공", "세부내용": "대한항공 KE123", "예약번호": "F789012", "비용": 1200000},
    {"항목ID": "I003", "여행ID": "T001", "종류": "관광", "세부내용": "에펠탑", "예약번호": "", "비용": 25000},
    {"항목ID": "I004", "여행ID": "T002", "종류": "숙소", "세부내용": "도쿄 프라자", "예약번호": "H345678", "비용": 500000},
    {"항목ID": "I005", "여행ID": "T002", "종류": "항공", "세부내용": "대한항공 KE345", "예약번호": "F901234", "비용": 350000}
]

travel_tips_data = [
    {"팁ID": "TP001", "목적지ID": "D001", "제목": "파리 여행 필수 정보", "카테고리": "일반", "내용": "파리에서는 지하철(메트로)이 가장 편리한 교통수단입니다. 관광객 대상 소매치기가 많으니 주의하세요."},
    {"팁ID": "TP002", "목적지ID": "D001", "제목": "파리 박물관 무료 입장일", "카테고리": "볼거리", "내용": "매월 첫 번째 일요일에는 루브르 박물관을 포함한 많은 박물관이 무료로 개방됩니다."},
    {"팁ID": "TP003", "목적지ID": "D002", "제목": "도쿄 지하철 이용법", "카테고리": "교통", "내용": "도쿄에서는 스이카(Suica) 또는 파스모(PASMO) 교통카드를 사용하면 편리합니다."},
    {"팁ID": "TP004", "목적지ID": "D002", "제목": "일본 온천 이용 에티켓", "카테고리": "문화", "내용": "온천에 들어가기 전에 반드시 몸을 씻어야 하며, 타투가 있는 경우 입장이 제한될 수 있습니다."},
    {"팁ID": "TP005", "목적지ID": "D003", "제목": "뉴욕 택시 이용 팁", "카테고리": "교통", "내용": "노란색 택시는 미터기를 반드시 사용하며, 팁은 요금의 15-20%가 일반적입니다."}
]

reviews_data = [
    {"리뷰ID": "R001", "목적지ID": "D001", "사용자ID": "U002", "평점": 5, "내용": "파리는 정말 낭만적인 도시예요. 에펠탑 야경이 특히 아름다웠습니다.", "작성일": "2024-03-15"},
    {"리뷰ID": "R002", "목적지ID": "D001", "사용자ID": "U003", "평점": 4, "내용": "루브르 박물관은 꼭 가볼 가치가 있어요. 하지만 너무 복잡하고 관광객이 많았습니다.", "작성일": "2024-02-20"},
    {"리뷰ID": "R003", "목적지ID": "D002", "사용자ID": "U001", "평점": 5, "내용": "도쿄는 깨끗하고 음식이 정말 맛있었어요. 다시 가고 싶은 도시입니다.", "작성일": "2024-01-10"},
    {"리뷰ID": "R004", "목적지ID": "D003", "사용자ID": "U004", "평점": 4, "내용": "뉴욕은 활기차고 다양한 문화를 경험할 수 있어서 좋았어요. 하지만 생각보다 많이 비쌌습니다.", "작성일": "2024-03-05"},
    {"리뷰ID": "R005", "목적지ID": "D004", "사용자ID": "U002", "평점": 5, "내용": "방콕은 음식이 맛있고 물가가 저렴해서 좋았어요. 왕궁과 사원들도 아름다웠습니다.", "작성일": "2024-02-28"}
]

weather_data = [
    {"날씨ID": "W001", "목적지ID": "D001", "월": 1, "평균기온": 5, "강수량": 18, "특징": "겨울철로 춥고 비가 자주 옵니다."},
    {"날씨ID": "W002", "목적지ID": "D001", "월": 4, "평균기온": 12, "강수량": 14, "특징": "봄철로 온화하지만 변덕스러운 날씨가 특징입니다."},
    {"날씨ID": "W003", "목적지ID": "D001", "월": 7, "평균기온": 20, "강수량": 22, "특징": "여름철로 따뜻하지만 소나기가 내릴 수 있습니다."},
    {"날씨ID": "W004", "목적지ID": "D001", "월": 10, "평균기온": 14, "강수량": 25, "특징": "가을철로 선선하고 단풍이 아름답습니다."},
    {"날씨ID": "W005", "목적지ID": "D002", "월": 1, "평균기온": 6, "강수량": 5, "특징": "겨울철로 춥고 건조합니다."}
]

# 데이터프레임 생성
destinations_df = pd.DataFrame(destinations_data)
attractions_df = pd.DataFrame(attractions_data)
hotels_df = pd.DataFrame(hotels_data)
flights_df = pd.DataFrame(flights_data)
packages_df = pd.DataFrame(packages_data)
user_trips_df = pd.DataFrame(user_trips_data)
trip_items_df = pd.DataFrame(trip_items_data)
travel_tips_df = pd.DataFrame(travel_tips_data)
reviews_df = pd.DataFrame(reviews_data)
weather_df = pd.DataFrame(weather_data)

#함수

In [4]:
# 1. 목적지 정보 조회 함수
def get_destination_info(destination_id: Optional[str] = None,
                         city: Optional[str] = None,
                         country: Optional[str] = None) -> List[Dict]:
    """
    목적지 정보를 조회합니다.

    Args:
        destination_id: 목적지 ID (예: D001, 선택사항)
        city: 도시 이름 (예: 파리, 선택사항)
        country: 국가 이름 (예: 프랑스, 선택사항)

    Returns:
        목적지 정보 목록
    """
    # 검색 조건이 하나도 제공되지 않은 경우 모든 목적지 반환
    if destination_id is None and city is None and country is None:
        return destinations_df.to_dict('records')

    result = destinations_df.copy()

    # ID로 필터링
    if destination_id:
        result = result[result['목적지ID'] == destination_id]

    # 도시로 필터링
    if city:
        result = result[result['도시'].str.contains(city, case=False)]

    # 국가로 필터링
    if country:
        result = result[result['국가'].str.contains(country, case=False)]

    if result.empty:
        return []

    return result

In [5]:
# 2. 관광지 정보 조회 함수
def get_attractions(destination_id: Optional[str] = None,
                    attraction_type: Optional[str] = None,
                    min_rating: Optional[int] = None) -> List[Dict]:
    """
    관광지 정보를 조회합니다.

    Args:
        destination_id: 목적지 ID (예: D001, 선택사항)
        attraction_type: 관광지 종류 (예: 랜드마크, 박물관, 선택사항)
        min_rating: 최소 인기도 (1-5, 선택사항)

    Returns:
        관광지 정보 목록
    """
    result = attractions_df.copy()

    # 목적지로 필터링
    if destination_id:
        result = result[result['목적지ID'] == destination_id]

    # 종류로 필터링
    if attraction_type:
        result = result[result['종류'] == attraction_type]

    # 인기도로 필터링
    if min_rating:
        result = result[result['인기도'] >= min_rating]

    if result.empty:
        return []

    # 목적지 정보 추가
    destinations_info = {}
    for idx, row in destinations_df.iterrows():
        destinations_info[row['목적지ID']] = {
            '도시': row['도시'],
            '국가': row['국가']
        }

    # 결과에 도시와 국가 정보 추가
    rich_result = []
    for attraction in result.to_dict('records'):
        dest_id = attraction['목적지ID']
        if dest_id in destinations_info:
            attraction['도시'] = destinations_info[dest_id]['도시']
            attraction['국가'] = destinations_info[dest_id]['국가']
        rich_result.append(attraction)

    return rich_result

In [6]:
# 3. 숙소 검색 함수
def search_hotels(destination_id: Optional[str] = None,
                 price_range: Optional[str] = None,
                 min_rating: Optional[float] = None,
                 facilities: Optional[str] = None) -> List[Dict]:
    """
    숙소를 검색합니다.

    Args:
        destination_id: 목적지 ID (예: D001, 선택사항)
        price_range: 가격대 (저가, 중가, 고가, 선택사항)
        min_rating: 최소 평점 (0-5, 선택사항)
        facilities: 필요한 편의시설 (쉼표로 구분, 예: 수영장,와이파이, 선택사항)

    Returns:
        숙소 정보 목록
    """
    result = hotels_df.copy()

    # 목적지로 필터링
    if destination_id:
        result = result[result['목적지ID'] == destination_id]

    # 가격대로 필터링
    if price_range:
        result = result[result['가격대'] == price_range]

    # 평점으로 필터링
    if min_rating:
        result = result[result['평점'] >= min_rating]

    # 편의시설로 필터링
    if facilities:
        facility_list = [f.strip() for f in facilities.split(',')]

        # 각 시설에 대해 확인
        for facility in facility_list:
            result = result[result['편의시설'].str.contains(facility, case=False)]

    if result.empty:
        return []

    # 목적지 정보 추가
    rich_result = []
    for hotel in result.to_dict('records'):
        dest_id = hotel['목적지ID']
        destination = destinations_df[destinations_df['목적지ID'] == dest_id]

        if not destination.empty:
            hotel['도시'] = destination.iloc[0]['도시']
            hotel['국가'] = destination.iloc[0]['국가']

        rich_result.append(hotel)

    return rich_result

# 4. 항공편 검색 함수
def search_flights(departure: str,
                  destination: str,
                  max_price: Optional[int] = None,
                  direct_only: Optional[bool] = False) -> List[Dict]:
    """
    항공편을 검색합니다.

    Args:
        departure: 출발지 (예: 서울)
        destination: 도착지 (예: 파리)
        max_price: 최대 가격 (선택사항)
        direct_only: 직항 여부 (선택사항, 기본값: False)

    Returns:
        항공편 정보 목록
    """
    result = flights_df[
        (flights_df['출발지'] == departure) &
        (flights_df['도착지'] == destination)
    ].copy()

    # 가격으로 필터링
    if max_price:
        result = result[result['가격'] <= max_price]

    # 직항 여부로 필터링
    if direct_only:
        result = result[result['경유'] == 0]

    if result.empty:
        return []

    return result.to_dict('records')

# 5. 여행 패키지 검색 함수
def search_packages(destination_id: Optional[str] = None,
                   min_days: Optional[int] = None,
                   max_days: Optional[int] = None,
                   max_price: Optional[int] = None) -> List[Dict]:
    """
    여행 패키지를 검색합니다.

    Args:
        destination_id: 목적지 ID (예: D001, 선택사항)
        min_days: 최소 일수 (선택사항)
        max_days: 최대 일수 (선택사항)
        max_price: 최대 가격 (선택사항)

    Returns:
        패키지 정보 목록
    """
    result = packages_df.copy()

    # 목적지로 필터링
    if destination_id:
        result = result[result['목적지ID'] == destination_id]

    # 최소 일수로 필터링
    if min_days:
        result = result[result['기간'] >= min_days]

    # 최대 일수로 필터링
    if max_days:
        result = result[result['기간'] <= max_days]

    # 최대 가격으로 필터링
    if max_price:
        result = result[result['가격'] <= max_price]

    if result.empty:
        return []

    # 목적지 정보 추가
    rich_result = []
    for package in result.to_dict('records'):
        dest_id = package['목적지ID']
        destination = destinations_df[destinations_df['목적지ID'] == dest_id]

        if not destination.empty:
            package['도시'] = destination.iloc[0]['도시']
            package['국가'] = destination.iloc[0]['국가']

        rich_result.append(package)

    return rich_result

# 6. 여행 계획 생성 함수
def create_trip_plan(user_id: str,
                    destination_id: str,
                    departure_date: str,
                    return_date: str,
                    budget: int) -> Dict:
    """
    새 여행 계획을 생성합니다.

    Args:
        user_id: 사용자 ID (예: U001)
        destination_id: 목적지 ID (예: D001)
        departure_date: 출발일 (YYYY-MM-DD 형식)
        return_date: 귀국일 (YYYY-MM-DD 형식)
        budget: 예산

    Returns:
        생성된 여행 계획 정보
    """
    # 목적지 확인
    destination = destinations_df[destinations_df['목적지ID'] == destination_id]
    if destination.empty:
        return {"error": "해당 목적지를 찾을 수 없습니다."}

    # 날짜 유효성 검사
    try:
        departure = datetime.strptime(departure_date, "%Y-%m-%d")
        return_day = datetime.strptime(return_date, "%Y-%m-%d")

        if departure >= return_day:
            return {"error": "귀국일은 출발일 이후여야 합니다."}
    except ValueError:
        return {"error": "날짜 형식이 올바르지 않습니다. YYYY-MM-DD 형식을 사용하세요."}

    # 새 여행 ID 생성
    last_trip_id = user_trips_df['여행ID'].iloc[-1] if not user_trips_df.empty else "T000"
    new_id = f"T{int(last_trip_id[1:]) + 1:03d}"

    # 여행 계획 생성
    new_trip = {
        "여행ID": new_id,
        "사용자ID": user_id,
        "목적지ID": destination_id,
        "출발일": departure_date,
        "귀국일": return_date,
        "예산": budget,
        "상태": "계획중"
    }

    # 여행 일수 계산
    days = (return_day - departure).days

    # 기본 정보 추가하여 응답 생성
    result = new_trip.copy()
    result['도시'] = destination.iloc[0]['도시']
    result['국가'] = destination.iloc[0]['국가']
    result['여행일수'] = days

    # 실제 구현에서는 DB에 저장
    # user_trips_df = user_trips_df.append(new_trip, ignore_index=True)

    return result

In [7]:
# 7. 여행 일정 항목 추가 함수
def add_trip_item(trip_id: str,
                 item_type: str,
                 details: str,
                 booking_number: Optional[str] = None,
                 cost: int = 0) -> Dict:
    """
    여행 계획에 항목을 추가합니다(숙소, 항공, 관광 등).

    Args:
        trip_id: 여행 ID (예: T001)
        item_type: 항목 종류 (예: 숙소, 항공, 관광, 식사, 교통)
        details: 항목 세부내용
        booking_number: 예약번호 (선택사항)
        cost: 비용 (선택사항)

    Returns:
        추가된 항목 정보
    """
    # 여행 계획 확인
    trip = user_trips_df[user_trips_df['여행ID'] == trip_id]
    if trip.empty:
        return {"error": "해당 여행 계획을 찾을 수 없습니다."}

    # 항목 종류 유효성 검사
    valid_types = ['숙소', '항공', '관광', '식사', '교통', '기타']
    if item_type not in valid_types:
        return {"error": f"유효하지 않은 항목 종류입니다. 가능한 항목: {', '.join(valid_types)}"}

    # 새 항목 ID 생성
    last_item_id = trip_items_df['항목ID'].iloc[-1] if not trip_items_df.empty else "I000"
    new_id = f"I{int(last_item_id[1:]) + 1:03d}"

    # 항목 생성
    new_item = {
        "항목ID": new_id,
        "여행ID": trip_id,
        "종류": item_type,
        "세부내용": details,
        "예약번호": booking_number if booking_number else "",
        "비용": cost
    }

    # 여행 정보 추가
    result = new_item.copy()
    result['출발일'] = trip.iloc[0]['출발일']
    result['귀국일'] = trip.iloc[0]['귀국일']

    # 목적지 정보 추가
    destination_id = trip.iloc[0]['목적지ID']
    destination = destinations_df[destinations_df['목적지ID'] == destination_id]
    if not destination.empty:
        result['도시'] = destination.iloc[0]['도시']
        result['국가'] = destination.iloc[0]['국가']

    # 실제 구현에서는 DB에 저장
    # trip_items_df = trip_items_df.append(new_item, ignore_index=True)

    return result

# 8. 여행 팁 조회 함수
def get_travel_tips(destination_id: Optional[str] = None,
                   category: Optional[str] = None) -> List[Dict]:
    """
    특정 목적지에 대한 여행 팁을 조회합니다.

    Args:
        destination_id: 목적지 ID (예: D001, 선택사항)
        category: 팁 카테고리 (예: 일반, 교통, 문화, 선택사항)

    Returns:
        여행 팁 목록
    """
    result = travel_tips_df.copy()

    # 목적지로 필터링
    if destination_id:
        result = result[result['목적지ID'] == destination_id]

    # 카테고리로 필터링
    if category:
        result = result[result['카테고리'] == category]

    if result.empty:
        return []

    # 목적지 정보 추가
    rich_result = []
    for tip in result.to_dict('records'):
        dest_id = tip['목적지ID']
        destination = destinations_df[destinations_df['목적지ID'] == dest_id]

        if not destination.empty:
            tip['도시'] = destination.iloc[0]['도시']
            tip['국가'] = destination.iloc[0]['국가']

        rich_result.append(tip)

    return rich_result

# 9. 목적지 날씨 정보 조회 함수
def get_weather_info(destination_id: str, month: Optional[int] = None) -> List[Dict]:
    """
    목적지의 월별 날씨 정보를 조회합니다.

    Args:
        destination_id: 목적지 ID (예: D001)
        month: 월 (1-12, 선택사항)

    Returns:
        날씨 정보 목록
    """
    result = weather_df[weather_df['목적지ID'] == destination_id].copy()

    # 월로 필터링
    if month and 1 <= month <= 12:
        result = result[result['월'] == month]

    if result.empty:
        return []

    # 목적지 정보 추가
    destination = destinations_df[destinations_df['목적지ID'] == destination_id]

    rich_result = []
    for weather in result.to_dict('records'):
        if not destination.empty:
            weather['도시'] = destination.iloc[0]['도시']
            weather['국가'] = destination.iloc[0]['국가']
        rich_result.append(weather)

    return rich_result

# 10. 여행 리뷰 조회 함수
def get_destination_reviews(destination_id: str, min_rating: Optional[int] = None) -> List[Dict]:
    """
    목적지에 대한 여행자 리뷰를 조회합니다.

    Args:
        destination_id: 목적지 ID (예: D001)
        min_rating: 최소 평점 (1-5, 선택사항)

    Returns:
        리뷰 목록
    """
    result = reviews_df[reviews_df['목적지ID'] == destination_id].copy()

    # 평점으로 필터링
    if min_rating and 1 <= min_rating <= 5:
        result = result[result['평점'] >= min_rating]

    if result.empty:
        return []

    # 목적지 정보 추가
    destination = destinations_df[destinations_df['목적지ID'] == destination_id]

    rich_result = []
    for review in result.to_dict('records'):
        if not destination.empty:
            review['도시'] = destination.iloc[0]['도시']
            review['국가'] = destination.iloc[0]['국가']
        rich_result.append(review)

    return rich_result

# 11. 사용자 여행 목록 조회 함수
def get_user_trips(user_id: str, status: Optional[str] = None) -> List[Dict]:
    """
    사용자의 여행 계획 목록을 조회합니다.

    Args:
        user_id: 사용자 ID (예: U001)
        status: 여행 상태 (예: 계획중, 완료, 선택사항)

    Returns:
        여행 계획 목록
    """
    result = user_trips_df[user_trips_df['사용자ID'] == user_id].copy()

    # 상태로 필터링
    if status:
        result = result[result['상태'] == status]

    if result.empty:
        return []

    # 목적지 정보와 항목 수 추가
    rich_result = []
    for trip in result.to_dict('records'):
        # 목적지 정보 추가
        dest_id = trip['목적지ID']
        destination = destinations_df[destinations_df['목적지ID'] == dest_id]

        if not destination.empty:
            trip['도시'] = destination.iloc[0]['도시']
            trip['국가'] = destination.iloc[0]['국가']

        # 여행 항목 수 계산
        trip_id = trip['여행ID']
        items = trip_items_df[trip_items_df['여행ID'] == trip_id]
        trip['항목수'] = len(items)

        # 여행 일수 계산
        departure = datetime.strptime(trip['출발일'], "%Y-%m-%d")
        return_day = datetime.strptime(trip['귀국일'], "%Y-%m-%d")
        trip['여행일수'] = (return_day - departure).days

        rich_result.append(trip)

    return rich_result

# 12. 여행 예산 분석 함수
def analyze_trip_budget(trip_id: str) -> Dict:
    """
    여행 예산을 분석합니다.

    Args:
        trip_id: 여행 ID (예: T001)

    Returns:
        예산 분석 정보
    """
    # 여행 정보 확인
    trip = user_trips_df[user_trips_df['여행ID'] == trip_id]
    if trip.empty:
        return {"error": "해당 여행 계획을 찾을 수 없습니다."}

    # 항목 정보 가져오기
    items = trip_items_df[trip_items_df['여행ID'] == trip_id]

    # 예산 정보
    total_budget = trip.iloc[0]['예산']
    spent_amount = items['비용'].sum() if not items.empty else 0
    remaining_budget = total_budget - spent_amount

    # 항목별 지출 분석
    item_types = {}
    if not items.empty:
        for _, item in items.iterrows():
            item_type = item['종류']
            cost = item['비용']

            if item_type in item_types:
                item_types[item_type] += cost
            else:
                item_types[item_type] = cost

    # 결과 생성
    result = {
        "여행ID": trip_id,
        "총예산": total_budget,
        "지출금액": spent_amount,
        "남은예산": remaining_budget,
        "예산사용률": round((spent_amount / total_budget) * 100, 1) if total_budget > 0 else 0,
        "항목별지출": [{"종류": k, "금액": v, "비율": round((v / spent_amount) * 100, 1) if spent_amount > 0 else 0} for k, v in item_types.items()]
    }

    # 목적지 정보 추가
    destination_id = trip.iloc[0]['목적지ID']
    destination = destinations_df[destinations_df['목적지ID'] == destination_id]
    if not destination.empty:
        result['도시'] = destination.iloc[0]['도시']
        result['국가'] = destination.iloc[0]['국가']

    return result

In [8]:
# OpenAI 함수 호출 정의
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_destination_info",
            "description": "목적지 정보를 조회합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "destination_id": {
                        "type": "string",
                        "description": "목적지 ID (예: D001, 선택사항)"
                    },
                    "city": {
                        "type": "string",
                        "description": "도시 이름 (예: 파리, 선택사항)"
                    },
                    "country": {
                        "type": "string",
                        "description": "국가 이름 (예: 프랑스, 선택사항)"
                    }
                }
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_attractions",
            "description": "관광지 정보를 조회합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "destination_id": {
                        "type": "string",
                        "description": "목적지 ID (예: D001, 선택사항)"
                    },
                    "attraction_type": {
                        "type": "string",
                        "description": "관광지 종류 (예: 랜드마크, 박물관, 선택사항)"
                    },
                    "min_rating": {
                        "type": "integer",
                        "description": "최소 인기도 (1-5, 선택사항)"
                    }
                }
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_hotels",
            "description": "숙소를 검색합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "destination_id": {
                        "type": "string",
                        "description": "목적지 ID (예: D001, 선택사항)"
                    },
                    "price_range": {
                        "type": "string",
                        "description": "가격대 (저가, 중가, 고가, 선택사항)"
                    },
                    "min_rating": {
                        "type": "number",
                        "description": "최소 평점 (0-5, 선택사항)"
                    },
                    "facilities": {
                        "type": "string",
                        "description": "필요한 편의시설 (쉼표로 구분, 예: 수영장,와이파이, 선택사항)"
                    }
                }
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_flights",
            "description": "항공편을 검색합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "departure": {
                        "type": "string",
                        "description": "출발지 (예: 서울)"
                    },
                    "destination": {
                        "type": "string",
                        "description": "도착지 (예: 파리)"
                    },
                    "max_price": {
                        "type": "integer",
                        "description": "최대 가격 (선택사항)"
                    },
                    "direct_only": {
                        "type": "boolean",
                        "description": "직항 여부 (선택사항, 기본값: False)"
                    }
                },
                "required": ["departure", "destination"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_packages",
            "description": "여행 패키지를 검색합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "destination_id": {
                        "type": "string",
                        "description": "목적지 ID (예: D001, 선택사항)"
                    },
                    "min_days": {
                        "type": "integer",
                        "description": "최소 일수 (선택사항)"
                    },
                    "max_days": {
                        "type": "integer",
                        "description": "최대 일수 (선택사항)"
                    },
                    "max_price": {
                        "type": "integer",
                        "description": "최대 가격 (선택사항)"
                    }
                }
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "create_trip_plan",
            "description": "새 여행 계획을 생성합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "user_id": {
                        "type": "string",
                        "description": "사용자 ID (예: U001)"
                    },
                    "destination_id": {
                        "type": "string",
                        "description": "목적지 ID (예: D001)"
                    },
                    "departure_date": {
                        "type": "string",
                        "description": "출발일 (YYYY-MM-DD 형식)"
                    },
                    "return_date": {
                        "type": "string",
                        "description": "귀국일 (YYYY-MM-DD 형식)"
                    },
                    "budget": {
                        "type": "integer",
                        "description": "예산"
                    }
                },
                "required": ["user_id", "destination_id", "departure_date", "return_date", "budget"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "add_trip_item",
            "description": "여행 계획에 항목을 추가합니다(숙소, 항공, 관광 등).",
            "parameters": {
                "type": "object",
                "properties": {
                    "trip_id": {
                        "type": "string",
                        "description": "여행 ID (예: T001)"
                    },
                    "item_type": {
                        "type": "string",
                        "description": "항목 종류 (예: 숙소, 항공, 관광, 식사, 교통)"
                    },
                    "details": {
                        "type": "string",
                        "description": "항목 세부내용"
                    },
                    "booking_number": {
                        "type": "string",
                        "description": "예약번호 (선택사항)"
                    },
                    "cost": {
                        "type": "integer",
                        "description": "비용 (선택사항)"
                    }
                },
                "required": ["trip_id", "item_type", "details"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_travel_tips",
            "description": "특정 목적지에 대한 여행 팁을 조회합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "destination_id": {
                        "type": "string",
                        "description": "목적지 ID (예: D001, 선택사항)"
                    },
                    "category": {
                        "type": "string",
                        "description": "팁 카테고리 (예: 일반, 교통, 문화, 선택사항)"
                    }
                }
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather_info",
            "description": "목적지의 월별 날씨 정보를 조회합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "destination_id": {
                        "type": "string",
                        "description": "목적지 ID (예: D001)"
                    },
                    "month": {
                        "type": "integer",
                        "description": "월 (1-12, 선택사항)"
                    }
                },
                "required": ["destination_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_destination_reviews",
            "description": "목적지에 대한 여행자 리뷰를 조회합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "destination_id": {
                        "type": "string",
                        "description": "목적지 ID (예: D001)"
                    },
                    "min_rating": {
                        "type": "integer",
                        "description": "최소 평점 (1-5, 선택사항)"
                    }
                },
                "required": ["destination_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_user_trips",
            "description": "사용자의 여행 계획 목록을 조회합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "user_id": {
                        "type": "string",
                        "description": "사용자 ID (예: U001)"
                    },
                    "status": {
                        "type": "string",
                        "description": "여행 상태 (예: 계획중, 완료, 선택사항)"
                    }
                },
                "required": ["user_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "analyze_trip_budget",
            "description": "여행 예산을 분석합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "trip_id": {
                        "type": "string",
                        "description": "여행 ID (예: T001)"
                    }
                },
                "required": ["trip_id"]
            }
        }
    }
]

In [9]:
def execute_function(function_name, arguments):
    function_map = {
        "get_destination_info": get_destination_info,
        "get_attractions": get_attractions,
        "search_hotels": search_hotels,
        "search_flights": search_flights,
        "search_packages": search_packages,
        "create_trip_plan": create_trip_plan,
        "add_trip_item": add_trip_item,
        "get_travel_tips": get_travel_tips,
        "get_weather_info": get_weather_info,
        "get_destination_reviews": get_destination_reviews,
        "get_user_trips": get_user_trips,
        "analyze_trip_budget": analyze_trip_budget
    }

    if function_name not in function_map:
        return {"error": f"Function {function_name} not found"}

    try:
        # user_id가 비어있는지 확인
        if "user_id" in arguments and (arguments["user_id"] is None or arguments["user_id"] == ""):
            arguments["user_id"] = "U001"  # 기본값 설정

        # 함수 실행
        result = function_map[function_name](**arguments)

        # 직렬화 가능하게 변환
        if isinstance(result, pd.DataFrame):
            return result.to_dict('records')

        if isinstance(result, list):
            serialized_list = []
            for item in result:
                if isinstance(item, pd.DataFrame):
                    serialized_list.append(item.to_dict('records'))
                elif isinstance(item, dict):
                    serialized_dict = {}
                    for k, v in item.items():
                        if isinstance(v, pd.DataFrame):
                            serialized_dict[k] = v.to_dict('records')
                        elif hasattr(v, 'item'):  # numpy 타입
                            serialized_dict[k] = v.item()
                        else:
                            serialized_dict[k] = v
                    serialized_list.append(serialized_dict)
                else:
                    serialized_list.append(item)
            return serialized_list

        if isinstance(result, dict):
            serialized_dict = {}
            for k, v in result.items():
                if isinstance(v, pd.DataFrame):
                    serialized_dict[k] = v.to_dict('records')
                elif hasattr(v, 'item'):  # numpy 타입
                    serialized_dict[k] = v.item()
                else:
                    serialized_dict[k] = v
            return serialized_dict

        # numpy 타입 처리
        if hasattr(result, 'item'):
            return result.item()

        return result
    except Exception as e:
        return {"error": str(e), "details": f"Error in {function_name} with args {arguments}"}

#실행


In [62]:
# OpenAI 클라이언트 설정
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", ""))

In [54]:
# 대화 기록을 저장할 변수
conversation_history = {}

In [55]:
def pch(user_id, text):
  print()
  print(f'=========={text}==========')

  for conversation in conversation_history[user_id]:
    print(conversation)

  print('=='*20)
  print()

In [58]:
def process_message(message, user_id, conversation_id=None):
    global conversation_history

    # 새로운 대화인 경우 대화 ID 생성 및 초기화
    if conversation_id is None or conversation_id not in conversation_history:
        conversation_id = f"conv_{len(conversation_history) + 1}"
        conversation_history[conversation_id] = [
            # 시스템 메시지로 사용자 ID 컨텍스트 설정
            {"role": "system", "content": f"현재 사용자 ID는 {user_id}입니다. 이 사용자의 정보를 조회할 때는 항상 이 ID를 사용하세요. 당신은 여행 계획 도우미입니다. 목적지 정보, 관광지, 숙소, 항공편, 여행 팁 등의 정보를 제공하고, 여행 계획 생성 및 관리를 도와주세요. 친절하고 유용한 답변을 제공하세요."}
        ]

    # 사용자 메시지 추가
    conversation_history[conversation_id].append({"role": "user", "content": message})

    pch(conversation_id, '시작 부분')

    # 메시지 목록 구성
    messages = conversation_history[conversation_id]

    try:
        # OpenAI API 호출
        response = client.chat.completions.create(
            model="gpt-4o",  # 또는 사용 가능한 최신 모델
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )

        # 응답 추출
        assistant_message = response.choices[0].message

        # 함수 호출이 필요한 경우
        if assistant_message.tool_calls:
            # 응답에 함수 호출 정보 저장
            conversation_history[conversation_id].append(assistant_message)

            pch(conversation_id, '도구 호출 인수 return')

            # 모든 함수 호출 수행
            for tool_call in assistant_message.tool_calls:
                function_name = tool_call.function.name
                function_args = json.loads(tool_call.function.arguments)

                # user_id가 명시되지 않은 경우 현재 사용자 ID 사용
                if "user_id" in function_args and not function_args["user_id"]:
                    function_args["user_id"] = user_id

                # 함수 실행 결과 가져오기
                function_response = execute_function(function_name, function_args)

                # 함수 결과를 대화 기록에 추가
                conversation_history[conversation_id].append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": json.dumps(function_response, ensure_ascii=False)
                })

            pch(conversation_id, '도구 호출 결과 send')

            # 함수 결과를 바탕으로 최종 응답 생성
            second_response = client.chat.completions.create(
                model="gpt-4o",
                messages=conversation_history[conversation_id]
            )

            final_response = second_response.choices[0].message.content

            # 최종 응답을 대화 기록에 추가
            conversation_history[conversation_id].append({
                "role": "assistant",
                "content": final_response
            })

            pch(conversation_id, '도구 호출 결과 참고 후 return')

            return final_response, conversation_id
        else:
            # 일반 응답인 경우
            content = assistant_message.content

            # 응답을 대화 기록에 추가
            conversation_history[conversation_id].append({
                "role": "assistant",
                "content": content
            })

            return content, conversation_id

    except Exception as e:
        error_message = f"오류가 발생했습니다: {str(e)}"
        return error_message, conversation_id

In [57]:
process_message('제주도 여행 패키지 검색해줘', 'U002', 'conv_1')


==========시작 부분==========
{'role': 'system', 'content': '현재 사용자 ID는 U002입니다. 이 사용자의 정보를 조회할 때는 항상 이 ID를 사용하세요. 당신은 여행 계획 도우미입니다. 목적지 정보, 관광지, 숙소, 항공편, 여행 팁 등의 정보를 제공하고, 여행 계획 생성 및 관리를 도와주세요. 친절하고 유용한 답변을 제공하세요.'}
{'role': 'user', 'content': '제주도 여행 패키지 검색해줘'}


==========도구 호출 정보 저장==========
{'role': 'system', 'content': '현재 사용자 ID는 U002입니다. 이 사용자의 정보를 조회할 때는 항상 이 ID를 사용하세요. 당신은 여행 계획 도우미입니다. 목적지 정보, 관광지, 숙소, 항공편, 여행 팁 등의 정보를 제공하고, 여행 계획 생성 및 관리를 도와주세요. 친절하고 유용한 답변을 제공하세요.'}
{'role': 'user', 'content': '제주도 여행 패키지 검색해줘'}
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_cXEq9pp8xlrfLf6wZLWJLWFH', function=Function(arguments='{"destination_id":"D002"}', name='search_packages'), type='function')])


==========도구 호출 결과 저장==========
{'role': 'system', 'content': '현재 사용자 ID는 U002입니다. 이 사용자의 정보를 조회할 때는 항상 이 ID를 사용하세요. 당신은 여행 계획 도우미입니다. 목적지 정보, 관광지, 숙소, 항공편, 여행 팁 등의 정

('죄송하지만, 제주도 여행 패키지 정보가 현재 시스템에 업데이트되지 않은 것 같습니다. 저 대신 여행사를 통해 최신 제주도 여행 패키지를 직접 확인해보시는 것도 좋을 것 같습니다.\n\n다른 여행 관련 질문이나 지원이 필요하시면 언제든지 알려주세요!',
 'conv_1')

In [59]:
process_message('내가 예약한건 있니?', 'U002', 'conv_1')


==========시작 부분==========
{'role': 'system', 'content': '현재 사용자 ID는 U002입니다. 이 사용자의 정보를 조회할 때는 항상 이 ID를 사용하세요. 당신은 여행 계획 도우미입니다. 목적지 정보, 관광지, 숙소, 항공편, 여행 팁 등의 정보를 제공하고, 여행 계획 생성 및 관리를 도와주세요. 친절하고 유용한 답변을 제공하세요.'}
{'role': 'user', 'content': '제주도 여행 패키지 검색해줘'}
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_cXEq9pp8xlrfLf6wZLWJLWFH', function=Function(arguments='{"destination_id":"D002"}', name='search_packages'), type='function')])
{'role': 'tool', 'tool_call_id': 'call_cXEq9pp8xlrfLf6wZLWJLWFH', 'name': 'search_packages', 'content': '[{"패키지ID": "P003", "목적지ID": "D002", "패키지명": "도쿄 2박3일 빠른여행", "가격": 800000, "기간": 3, "포함사항": "호텔,항공,관광", "제외사항": "식사,개인경비,팁", "인기도": 3, "도시": "도쿄", "국가": "일본"}, {"패키지ID": "P004", "목적지ID": "D002", "패키지명": "일본 도쿄 4박5일 맛집 투어", "가격": 1200000, "기간": 5, "포함사항": "호텔,항공,관광,식사 8회", "제외사항": "개인경비,팁", "인기도": 5, "도시": "도쿄", "국가": "일본"}]'}
{'ro

('현재 U002님의 예약된 여행은 없지만, 계획 중인 여행이 하나 있습니다. 계획 중인 여행의 상세 정보는 다음과 같습니다:\n\n- **목적지**: 뉴욕, 미국\n- **출발일**: 2025년 6월 10일\n- **귀국일**: 2025년 6월 16일\n- **예산**: 3,000,000원\n- **상태**: 계획중\n- **여행일수**: 6일\n\n여행 계획을 구체화하거나 다른 질문이 있으시다면 언제든지 말씀해 주세요!',
 'conv_1')

In [60]:
process_message('댄스파리 예에', 'U002', 'conv_1')


==========시작 부분==========
{'role': 'system', 'content': '현재 사용자 ID는 U002입니다. 이 사용자의 정보를 조회할 때는 항상 이 ID를 사용하세요. 당신은 여행 계획 도우미입니다. 목적지 정보, 관광지, 숙소, 항공편, 여행 팁 등의 정보를 제공하고, 여행 계획 생성 및 관리를 도와주세요. 친절하고 유용한 답변을 제공하세요.'}
{'role': 'user', 'content': '제주도 여행 패키지 검색해줘'}
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_cXEq9pp8xlrfLf6wZLWJLWFH', function=Function(arguments='{"destination_id":"D002"}', name='search_packages'), type='function')])
{'role': 'tool', 'tool_call_id': 'call_cXEq9pp8xlrfLf6wZLWJLWFH', 'name': 'search_packages', 'content': '[{"패키지ID": "P003", "목적지ID": "D002", "패키지명": "도쿄 2박3일 빠른여행", "가격": 800000, "기간": 3, "포함사항": "호텔,항공,관광", "제외사항": "식사,개인경비,팁", "인기도": 3, "도시": "도쿄", "국가": "일본"}, {"패키지ID": "P004", "목적지ID": "D002", "패키지명": "일본 도쿄 4박5일 맛집 투어", "가격": 1200000, "기간": 5, "포함사항": "호텔,항공,관광,식사 8회", "제외사항": "개인경비,팁", "인기도": 5, "도시": "도쿄", "국가": "일본"}]'}
{'ro

('뉴욕 여행을 계획 중이시군요! 뉴욕은 멋진 도시로, 많은 볼거리와 즐길 거리가 가득합니다. 그 여행을 준비하며 도움을 받을 수 있는 몇 가지 아이디어나 팁이 필요하신가요? 예를 들어 항공편, 숙소, 관광지 추천 등 무엇이든 도와드리겠습니다.',
 'conv_1')

In [61]:
process_message('뉴욕으로의 항공편 검색해줘', 'U002', 'conv_1')


==========시작 부분==========
{'role': 'system', 'content': '현재 사용자 ID는 U002입니다. 이 사용자의 정보를 조회할 때는 항상 이 ID를 사용하세요. 당신은 여행 계획 도우미입니다. 목적지 정보, 관광지, 숙소, 항공편, 여행 팁 등의 정보를 제공하고, 여행 계획 생성 및 관리를 도와주세요. 친절하고 유용한 답변을 제공하세요.'}
{'role': 'user', 'content': '제주도 여행 패키지 검색해줘'}
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_cXEq9pp8xlrfLf6wZLWJLWFH', function=Function(arguments='{"destination_id":"D002"}', name='search_packages'), type='function')])
{'role': 'tool', 'tool_call_id': 'call_cXEq9pp8xlrfLf6wZLWJLWFH', 'name': 'search_packages', 'content': '[{"패키지ID": "P003", "목적지ID": "D002", "패키지명": "도쿄 2박3일 빠른여행", "가격": 800000, "기간": 3, "포함사항": "호텔,항공,관광", "제외사항": "식사,개인경비,팁", "인기도": 3, "도시": "도쿄", "국가": "일본"}, {"패키지ID": "P004", "목적지ID": "D002", "패키지명": "일본 도쿄 4박5일 맛집 투어", "가격": 1200000, "기간": 5, "포함사항": "호텔,항공,관광,식사 8회", "제외사항": "개인경비,팁", "인기도": 5, "도시": "도쿄", "국가": "일본"}]'}
{'ro

('서울에서 뉴욕으로 가는 항공편 정보를 찾았습니다. 다음과 같은 항공편이 있습니다:\n\n- **항공사**: 대한항공\n- **출발시간**: 10:00\n- **도착시간**: 10:30 (현지 시간)\n- **비행시간**: 약 14시간\n- **경유**: 없음 (직항)\n- **가격**: 1,500,000원\n\n항공편 예약이나 추가 정보가 필요하시면 언제든지 말씀해 주세요!',
 'conv_1')